# Retrieval

Retrieval is the centerpiece of our retrieval augmented generation (RAG) flow.

Let's get our vectorDB from before.

## Vectorstore retrieval


In [ ]:
# import os
# import openai
# import sys
# sys.path.append('../..')

# from dotenv import load_dotenv, find_dotenv
# _ = load_dotenv(find_dotenv()) # read local .env file

# openai.api_key  = os.environ['OPENAI_API_KEY']

In [ ]:
!pip install -q lark tiktoken
!pip install -q chromadb langchain_chroma
!pip install -q langchain_huggingface langchain-community
!pip install -q pypdf PyPDF2
!pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 46.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.5/615.5 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.8/273.8 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.2/93.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 4.3 MB/s eta 0:00:00

### Similarity Search

In [ ]:
from langchain.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts import PromptTemplate
import torch
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          pipeline, BitsAndBytesConfig)
import PyPDF2
import os
from getpass import getpass

In [ ]:
os.environ["HF_TOKEN"] = "YOUR_HUGGINGFACE_TOKEN"

In [ ]:
def merge_pdfs(input_paths, output_path):
    merger = PyPDF2.PdfMerger()
    for path in input_paths:
        with open(path, 'rb') as file:
            merger.append(file)
    with open(output_path, 'wb') as output_file:
        merger.write(output_file)

In [ ]:
pdf_files = ["/content/machinelearning-lecture01.pdf",
             "/content/MachineLearning-Lecture02.pdf",
             "/content/MachineLearning-Lecture03.pdf"]

output_filename = "/content/merged.pdf"

merge_pdfs(pdf_files, output_filename)

print(f"Merged PDF saved to: {output_filename}")

Merged PDF saved to: /content/merged.pdf


In [ ]:
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=quantization_config)
tokenizer = AutoTokenizer.from_pretrained(model_id)

pipe = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.1,
    max_new_tokens=500,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=False,
)
llm = HuggingFacePipeline(pipeline=pipe)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

`low_cpu_mem_usage` was None, now set to True since model is quantized.


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

... (3 outputs left out)

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

In [ ]:
loader = PyPDFLoader("/content/merged.pdf")
docs = loader.load()

In [ ]:
chunk_size = 1500
chunk_overlap = 150
text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size,
                                               chunk_overlap=chunk_overlap,
                                               add_start_index = True)
splits = text_splitter.split_documents(docs)

In [ ]:
model_name = "BAAI/bge-m3"
embedding = HuggingFaceEmbeddings(model_name=model_name)
vectordb = Chroma.from_documents(documents=splits, embedding=embedding)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [ ]:
print(vectordb._collection.count())

151


In [ ]:
texts = [
    """The Amanita phalloides has a large and imposing epigeous (aboveground) fruiting body (basidiocarp).""",
    """A mushroom with a large fruiting body is the Amanita phalloides. Some varieties are all-white.""",
    """A. phalloides, a.k.a Death Cap, is one of the most poisonous of all known mushrooms.""",
]

In [ ]:
smalldb = Chroma.from_texts(texts, embedding=embedding)

In [ ]:
question = "Tell me about all-white mushrooms with large fruiting bodies"
smalldb.similarity_search(question, k=2)

[Document(metadata={'page': 3, 'source': '/content/merged.pdf', 'start_index': 2732}, page_content='principles. So I assume all of you know what big?O notation, that all of you know about \nsort of data structures like queues, stacks, binary trees, and that all of you know enough \nprogramming skills to, like, write a simple computer program. And it turns out that most'),
 Document(metadata={'page': 16, 'source': '/content/merged.pdf', 'start_index': 1350}, page_content="regions. And using that as a pre-processing step, they eventually built this sort of 3D \nmodel of Stanford campus in a single picture. You can sort of walk into the ceiling, look \naround the campus. Okay? This actually turned out to be a mix of supervised and \nunsupervised learning, but the unsupervised learning, this sort of clustering was the first \nstep.  \nSo it turns out these sorts of unsupervised — clustering algorithms are actually routinely \nused for many different problems, things like organizing computi

In [ ]:
smalldb.max_marginal_relevance_search(question,k=2, fetch_k=3)

[Document(metadata={'page': 3, 'source': '/content/merged.pdf', 'start_index': 2732}, page_content='principles. So I assume all of you know what big?O notation, that all of you know about \nsort of data structures like queues, stacks, binary trees, and that all of you know enough \nprogramming skills to, like, write a simple computer program. And it turns out that most'),
 Document(metadata={'page': 16, 'source': '/content/merged.pdf', 'start_index': 1350}, page_content="regions. And using that as a pre-processing step, they eventually built this sort of 3D \nmodel of Stanford campus in a single picture. You can sort of walk into the ceiling, look \naround the campus. Okay? This actually turned out to be a mix of supervised and \nunsupervised learning, but the unsupervised learning, this sort of clustering was the first \nstep.  \nSo it turns out these sorts of unsupervised — clustering algorithms are actually routinely \nused for many different problems, things like organizing computi

### Abordando a diversidade: relevância marginal máxima

Na última aula, apresentamos um problema: como impor diversidade nos resultados da pesquisa.

`Relevância marginal máxima` se esforça para atingir relevância para a consulta *e diversidade* entre os resultados.

In [ ]:
question = "what did they say about matlab?"
docs_ss = vectordb.similarity_search(question,k=3)
print(docs_ss[0].page_content[:100])
print(docs_ss[1].page_content[:100])

those homeworks will be done in either MATLAB or in Octave, which is sort of — I 
know some people c
into his office and he said, "Oh, professor, professor, thank you so much for your 
machine learning


Observe a diferença nos resultados com `MMR`.

In [ ]:
docs_mmr = vectordb.max_marginal_relevance_search(question,k=3)
print(docs_mmr[0].page_content[:100])
print(docs_mmr[1].page_content[:100])

those homeworks will be done in either MATLAB or in Octave, which is sort of — I 
know some people c
So even today, there are some people that say, well, computers can't do anything that 
they're not e


### Addressing Specificity: working with metadata

In last lecture, we showed that a question about the third lecture can include results from other lectures as well.

To address this, many vectorstores support operations on `metadata`.

`metadata` provides context for each embedded chunk.

In [ ]:
question = "We’ll then talk about a probable second interpretation of linear regression"

docs = vectordb.similarity_search(
    question,
    k=3,
    filter={"source":"/content/MachineLearning-Lecture03.pdf"}
)
for d in docs:
    print(d.metadata)

### Abordando a especificidade: trabalhando com metadados usando o recuperador de autoconsulta

Mas temos um desafio interessante: geralmente queremos inferir os metadados da própria consulta.

Para abordar isso, podemos usar `SelfQueryRetriever`, que usa um LLM para extrair:

1. A string `query` a ser usada para pesquisa de vetor
2. Um filtro de metadados para passar também

A maioria dos bancos de dados de vetor oferece suporte a filtros de metadados, então isso não requer nenhum novo banco de dados ou índices.

In [ ]:
# from langchain.llms import OpenAI
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain.chains.query_constructor.base import AttributeInfo

In [ ]:
metadata_field_info = [
    AttributeInfo(
        name="source",
        description="The lecture the chunk is from, should be one of `/content/machinelearning-lecture01.pdf`, `/content/MachineLearning-Lecture02.pdf`, or `/content/MachineLearning-Lecture03.pdf`",
        type="string",
    ),
    AttributeInfo(
        name="page",
        description="The page from the lecture",
        type="integer",
    ),
]

**Note:** The default model for `OpenAI` ("from langchain.llms import OpenAI") is `text-davinci-003`. Due to the deprication of OpenAI's model `text-davinci-003` on 4 January 2024, you'll be using OpenAI's recommended replacement model `gpt-3.5-turbo-instruct` instead.

In [ ]:
document_content_description = "Lecture notes"
# llm = OpenAI(model='gpt-3.5-turbo-instruct', temperature=0)

retriever = SelfQueryRetriever.from_llm(
    llm,
    vectordb,
    document_content_description,
    metadata_field_info,
)

In [ ]:
# retriever = vectordb.as_retriever(search_type = "similarity", search_kwargs={"k": 3})
retriever = vectordb.as_retriever(search_type = "mmr", search_kwargs={"k": 3})

In [ ]:
from langchain_core.runnables import RunnablePassthrough

In [ ]:
template= """
<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
Você é um assistente virtual prestativo e está respondendo perguntas gerais.
Use os seguintes pedaços de contexto recuperado para responder à pergunta.
Se você não sabe a resposta, apenas diga que não sabe. Mantenha a resposta concisa.
<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Pergunta: {pergunta}
Contexto: {contexto}
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""

prompt = PromptTemplate(
    input_variables=["contexto", "pergunta"],
    template=template)

rag = ({"contexto": retriever, "pergunta": RunnablePassthrough()}
       | prompt | llm | StrOutputParser())

In [ ]:
question = "what did they say about regression in the third lecture?"
rag.invoke(question)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


'According to the provided context, there is no specific discussion about regression in the third lecture. The instructor mentions linear regression in the introduction, but does not elaborate further on it.'

In [ ]:
question = "what did they say about regression in the third lecture?"
docs = retriever.invoke(question)
for d in docs:
    print(d.metadata)

{'page': 40, 'source': '/content/merged.pdf', 'start_index': 0}
{'page': 22, 'source': '/content/merged.pdf', 'start_index': 1347}
{'page': 8, 'source': '/content/merged.pdf', 'start_index': 1352}


**You will receive a warning** about predict_and_parse being deprecated the first time you executing the next line. This can be safely ignored.

### Truques adicionais: compressão

Outra abordagem para melhorar a qualidade dos documentos recuperados é a compressão.

As informações mais relevantes para uma consulta podem estar enterradas em um documento com muito texto irrelevante.

Passar esse documento completo pelo seu aplicativo pode levar a chamadas LLM mais caras e respostas mais pobres.

A compressão contextual visa corrigir isso.

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

In [ ]:
def pretty_print_docs(docs):
    print(f"\n{'-' * 100}\n".join([f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]))

In [ ]:
# Wrap our vectorstore
# llm = OpenAI(temperature=0, model="gpt-3.5-turbo-instruct")
compressor = LLMChainExtractor.from_llm(llm)

In [ ]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectordb.as_retriever()
)

In [ ]:
question = "what did they say about matlab?"
compressed_docs = compression_retriever.invoke(question)
pretty_print_docs(compressed_docs)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Document 1:

"MATLAB is I guess part of the programming language that makes it very easy to write codes using matrices, to write code for numerical routines, to move data around, to plot data." 
"And it's sort of an extremely easy to learn tool to use for implementing a lot of learning algorithms." 
"So yeah, just a side comment for those of you that haven't seen MATLAB before..." 
"MATLAB is... sort of an extremely easy to learn tool to use..."

NO_OUTPUT.  The provided context does not contain information about what someone said about Matlab. It contains a lecture about using Matlab or Octave for homework assignments. The lecturer describes Matlab as being easy to learn and use, but does not provide specific comments or quotes about Matlab. Therefore, no output can be generated based on the given context.
----------------------------------------------------------------------------------------------------
Document 2:

NO_OUTPUT. The given context does not contain any information relat

## Combining various techniques

In [ ]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectordb.as_retriever(search_type="mmr")
)

In [ ]:
question = "what did they say about matlab?"
compressed_docs = compression_retriever.get_relevant_documents(question)
pretty_print_docs(compressed_docs)

<ipython-input-108-04d3a1efc988>:2: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  compressed_docs = compression_retriever.get_relevant_documents(question)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Document 1:

"MATLAB", "Octave", "free version of MATLAB", "programming language", "easy to learn tool", "numerical routines", "plot data". 

NO_OUTPUT.  The provided context does not contain information about what was said about Matlab. It only mentions its existence and usage in the class. There are no specific comments or opinions about Matlab itself.  Therefore, we cannot extract any relevant parts related to what was said about Matlab.  If you need more context or information, please provide it.  Otherwise, I'm happy to help with any other questions!
----------------------------------------------------------------------------------------------------
Document 2:

NO_OUTPUT.  No mention of MATLAB in the given context.  The context only talks about machine learning, logistic regression, PCA, data networks, and a brief mention of a MATLAB tutorial, but no actual information about MATLAB itself.  Therefore, there is no relevant information to extract.  Return NO_OUTPUT.  NO_OUTPUT.  NO

## Outros tipos de recuperação

Vale a pena notar que o vectordb não é o único tipo de ferramenta para recuperar documentos.

A abstração do recuperador `LangChain` inclui outras maneiras de recuperar documentos, como TF-IDF ou SVM.

In [ ]:
from langchain.retrievers import SVMRetriever
from langchain.retrievers import TFIDFRetriever
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
# Load PDF
loader = PyPDFLoader("docs/cs229_lectures/MachineLearning-Lecture01.pdf")
pages = loader.load()
all_page_text=[p.page_content for p in pages]
joined_page_text=" ".join(all_page_text)

# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1500,chunk_overlap = 150)
splits = text_splitter.split_text(joined_page_text)

In [ ]:
# Retrieve
svm_retriever = SVMRetriever.from_texts(splits,embedding)
tfidf_retriever = TFIDFRetriever.from_texts(splits)

In [ ]:
question = "What are major topics for this class?"
docs_svm=svm_retriever.get_relevant_documents(question)
docs_svm[0]

In [ ]:
question = "what did they say about matlab?"
docs_tfidf=tfidf_retriever.get_relevant_documents(question)
docs_tfidf[0]